In [17]:
#r "bin/Debug/net10.0/task17.dll"
#r "nuget: ScottPlot, 5.0.54"
using System;
using System.Collections.Generic;
using ScottPlot;
using task17;


public class TestCommand : ICommand 
{
    private readonly int _id;
    private int _counter = 0;
    public int Id => _id;
    public int Counter => _counter;

    public TestCommand(int id) => _id = id;

    public void Execute() => Console.WriteLine($"Поток {_id} вызов {++_counter}");
}

Installed Packages ScottPlot, 5.0.54

In [18]:

var scheduler = new RoundRobin();
var handler = new ExceptionHandler();
var serverThread = new ServerThread(handler, scheduler);
string reportText = "";
var history = new List<(int iteration, int id, int counter)>();
int iteration = 0;
for (int i = 1; i <= 5; i++) {
    scheduler.Add(new TestCommand(i));
}

while (scheduler.HasCommand()) {
    var cmd = (TestCommand)scheduler.Select();
    cmd.Execute();
    iteration++;
    history.Add((iteration, cmd.Id, cmd.Counter));
    reportText += $"Итерация: Поток {cmd.Id}, вызов {cmd.Counter}\n";
    
    if (cmd.Counter < 3) {
        scheduler.Add(cmd);
    }
}


var hardStop = new HardStopCommand(serverThread);
serverThread.QueueCommand(hardStop);


File.WriteAllText("report.txt", reportText);

Поток 1 вызов 1
Поток 2 вызов 1
Поток 3 вызов 1
Поток 4 вызов 1
Поток 5 вызов 1
Поток 1 вызов 2
Поток 2 вызов 2
Поток 3 вызов 2
Поток 4 вызов 2
Поток 5 вызов 2
Поток 1 вызов 3
Поток 2 вызов 3
Поток 3 вызов 3
Поток 4 вызов 3
Поток 5 вызов 3


In [19]:
var plt = new ScottPlot.Plot(); 

foreach (var id in Enumerable.Range(1, 5)) {
    var data = history.Where(h => h.id == id).ToList();
    var x = data.Select(h => (double)h.iteration).ToArray();
    var y = data.Select(h => (double)h.counter).ToArray();
    
    var scatter = plt.Add.Scatter(x, y);
    scatter.Label = $"Поток {id}";
}

plt.Title("Выполнение длительных операций");
plt.XLabel("Номер итерации");
plt.YLabel("Счетчик вызовов");

plt.ShowLegend();

plt.SavePng("plt.png", 800, 600);


(9,5): warning CS0618: "Scatter.Label" является устаревшим: 'use LegendText'

